# 额外的周末练习 - 第 2 周

现在，使用您从第 2 周学到的所有知识为您在第 1 周练习中构建的技术问题/回答器构建完整的原型。

这应该包括 Gradio UI、流媒体、使用系统提示来添加专业知识以及在模型之间切换的能力。如果您能够演示工具的使用，则可获得奖励积分！

如果您觉得大胆，请看看是否可以添加音频输入，以便您可以与它交谈，并让它用音频进行响应。 ChatGPT 或 Claude 可以帮助您，如果您有疑问，也可以给我发电子邮件。

我很快就会在这里发布完整的解决方案 - 除非有人比我先一步......

这方面的商业应用有很多，从语言导师到公司入职解决方案，再到人工智能伴侣和课程（就像这个！），我迫不及待地想看到你的结果。

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
import os
import json
import gradio as gr
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_contents

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
else:
    print("API key founded")



In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
MODEL = "gpt-oss:20b"
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
system_prompt = """
Role: You are an expert Technical Recruiter and Job Analyst.

Task: Analyze the provided Job Description (JD) and extract all required skills (technical, soft skills, and tools). Focus on the required skill not company's benefits.

Requirements: > 1. Rank the skills in a numbered list from 1 (Most Critical) to N (Least Critical).
2. Ranking Logic: Base the rank on the frequency of mention, the "Required" vs. "Preferred" sections, and how central the skill is to the core responsibilities described.
3. Provide a brief (one-sentence) justification for why the top 3 skills were ranked highest.
"""

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
crawl_function = {
    "name": "fetch_website_contents",
    "description": "Receive a website url and then return content of that website.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "An URL of the website that user want to get content",
            },
        },
        "required": ["url"],
        "additionalProperties": False
    }
}

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
tools = [
    {"type": "function", "function": crawl_function}
]

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]}for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content
        

In [ ]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function_name == "fetch_website_contents":
            arguments = json.load(tool_call.function.arguments)
            url = arguments.get("url")
            response_content = fetch_website_content(url)
            resoponses.append({
                "role": "tool",
                "content": response_content,
                
            })
    
    return responses
    

In [ ]:
gr.ChatInterface(fn=chat).launch()